# INC-001 Minute-grain profiling: `steps.json` and `heart_rate.json`

Read-only; proposes, decides nothing. Each participant file is read, profiled and released before the next one
(never all 50 in memory). Only counts, aggregates and participant/timestamp identifiers are printed.
Cell ids (`shr-NN`) are referenced from the findings document.
Grid: 28 days x 1440 minutes = 40,320 slots starting at 2026-01-08T00:00 local (UTC-05:00).

In [1]:
import json, re, time, collections, hashlib
from datetime import date, datetime, timedelta
from pathlib import Path

T0 = time.perf_counter()
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "participants").is_dir())
EVENTS = ROOT / "data" / "wearable_events"
GRID_START = datetime(2026, 1, 8, 0, 0)
N_SLOTS = 28 * 1440
ORD0 = date(2026, 1, 8).toordinal()
UTC_OFFSET_MIN = -5 * 60          # fixed study offset: local = UTC - 5h
TS = re.compile(r"^(\d{4})-(\d\d)-(\d\d)T(\d\d):(\d\d):(\d\d)(\.\d+)?(Z|[+-]\d\d:\d\d)?$")
participants = sorted(p.name for p in EVENTS.iterdir() if p.is_dir())

def show(label, value, examples=None):
    extra = f"  examples={sorted(examples)[:5]}" if examples else ""
    print(f"{label}: {value}{extra}")

def slot_to_ts(slot):
    return (GRID_START + timedelta(minutes=slot)).strftime("%Y-%m-%dT%H:%M")

def profile(pid, fname, vkey):
    """Profile one participant file; return a small aggregate dict (no raw rows retained)."""
    t = time.perf_counter()
    raw = (EVENTS / pid / fname).read_bytes()
    t_read = time.perf_counter() - t
    t = time.perf_counter()
    rows = json.loads(raw)
    t_parse = time.perf_counter() - t
    a = collections.Counter()
    a["bytes"] = len(raw)
    a["records"] = len(rows)
    seen = bytearray(N_SLOTS)
    z_label, z_conv, other_fmt = [], [], []
    values = collections.Counter()
    prev_slot = None
    hr_runs = []                     # (start_slot, length, value) for runs >= 10 (heart rate only)
    run_val, run_start, run_len = None, None, 0
    keysets = collections.Counter()
    ts_hash = hashlib.md5()
    for i, r in enumerate(rows):
        keysets[tuple(sorted(r))] += 1
        if r.get("participant_id") != pid:
            a["participant_id != folder"] += 1
        ts = r.get("timestamp")
        ts_hash.update(str(ts).encode())
        m = TS.match(ts) if isinstance(ts, str) else None
        if not m:
            a["timestamp unparseable/missing"] += 1
            other_fmt.append(i)
            slot = None
        else:
            y, mo, d, hh, mi, ss, frac, tz = m.groups()
            if frac:
                a["timestamp has fractional seconds"] += 1
            if ss != "00":
                a["timestamp non-zero seconds"] += 1
            try:
                day_off = date(int(y), int(mo), int(d)).toordinal() - ORD0
            except ValueError:
                a["timestamp invalid calendar date"] += 1
                day_off = None
            if day_off is None:
                slot = None
            else:
                base = day_off * 1440 + int(hh) * 60 + int(mi)
                if tz == "Z":
                    a["timestamp UTC 'Z' suffix"] += 1
                    z_label.append(base)
                    z_conv.append(base + UTC_OFFSET_MIN)
                    slot = None
                elif tz:
                    a["timestamp explicit offset (not Z)"] += 1
                    slot = None
                else:
                    a["timestamp naive"] += 1
                    slot = base
        if slot is not None:
            if not 0 <= slot < N_SLOTS:
                a["naive timestamp outside study grid"] += 1
            else:
                if seen[slot]:
                    a["duplicate minute slot (naive)"] += 1
                seen[slot] = 1
            if prev_slot is not None and slot < prev_slot:
                a["out-of-order record"] += 1
            prev_slot = slot
        v = r.get(vkey)
        if v is None:
            a["value null/missing"] += 1
        elif isinstance(v, bool) or not isinstance(v, int):
            a["value not an integer"] += 1
        else:
            values[v] += 1
            if vkey == "heart_rate_bpm":
                if v == run_val:
                    run_len += 1
                else:
                    if run_len >= 10:
                        hr_runs.append((run_start, run_len, run_val))
                    run_val, run_start, run_len = v, slot, 1
    if vkey == "heart_rate_bpm" and run_len >= 10:
        hr_runs.append((run_start, run_len, run_val))
    a["naive slots filled"] = sum(seen)
    a["naive grid gaps"] = N_SLOTS - sum(seen)
    # Interpretation of Z records (observations only): do they land in empty or occupied naive slots?
    for label, slots in (("as_label", z_label), ("utc_converted", z_conv)):
        for s in slots:
            if not 0 <= s < N_SLOTS:
                a[f"Z {label}: outside grid"] += 1
            elif seen[s]:
                a[f"Z {label}: collides with naive record"] += 1
            else:
                a[f"Z {label}: fills an empty slot"] += 1
    if values:
        a["min"], a["max"] = min(values), max(values)
    return {"a": a, "values": values, "hr_runs": hr_runs, "t_read": t_read, "t_parse": t_parse,
            "ts_hash": ts_hash.hexdigest(), "keysets": keysets, "z_conv": z_conv}

print("participants:", len(participants), "| grid slots per file:", N_SLOTS)

participants: 50 | grid slots per file: 40320


## 1. Profile all files (one participant at a time; aggregates cached for later cells)

In [2]:
RES = {"steps": {}, "heart_rate": {}}
VKEY = {"steps": "steps", "heart_rate": "heart_rate_bpm"}
for pid in participants:
    for kind in ("steps", "heart_rate"):
        res = profile(pid, f"{kind}.json", VKEY[kind])
        RES[kind][pid] = res
print("profiling pass done; elapsed", round(time.perf_counter() - T0, 1), "s")
for kind in RES:
    rs = RES[kind].values()
    rd, ps = sum(r["t_read"] for r in rs), sum(r["t_parse"] for r in rs)
    nrec = sum(r["a"]["records"] for r in rs)
    print(f"{kind}.json: files={len(RES[kind])} records={nrec:,} bytes={sum(r['a']['bytes'] for r in rs):,} "
          f"read={rd:.2f}s json-parse={ps:.2f}s (mean per file read {1000*rd/len(RES[kind]):.0f} ms, parse {1000*ps/len(RES[kind]):.0f} ms; "
          f"parse {1e6*ps/nrec:.2f} microseconds per minute-record)")
    print("  key sets:", dict(sum((r["keysets"] for r in rs), collections.Counter())))

profiling pass done; elapsed 37.4 s
steps.json: files=50 records=2,014,950 bytes=193,841,867 read=0.03s json-parse=0.66s (mean per file read 1 ms, parse 13 ms; parse 0.33 microseconds per minute-record)
  key sets: {('participant_id', 'steps', 'timestamp'): 1893987, ('_unit', 'participant_id', 'steps', 'timestamp'): 120963}
heart_rate.json: files=50 records=2,014,950 bytes=209,633,824 read=0.03s json-parse=0.65s (mean per file read 1 ms, parse 13 ms; parse 0.32 microseconds per minute-record)
  key sets: {('heart_rate_bpm', 'participant_id', 'timestamp'): 2014950}


## 2. Records per participant vs 40,320; duplicates and gaps on the minute grid

In [3]:
for kind in RES:
    A = {p: RES[kind][p]["a"] for p in participants}
    print(f"--- {kind}")
    print("records per participant histogram (n->participants):", dict(sorted(collections.Counter(a["records"] for a in A.values()).items())))
    off = {p: a["records"] for p, a in A.items() if a["records"] != N_SLOTS}
    show("participants with records != 40320", len(off), [f"{p}={n}" for p, n in off.items()])
    for key in ("duplicate minute slot (naive)", "naive grid gaps", "out-of-order record", "naive timestamp outside study grid"):
        hit = {p: a[key] for p, a in A.items() if a[key]}
        show(f"{key}: records/slots={sum(hit.values())} participants", len(hit), [f"{p}={n}" for p, n in hit.items()])
    nfill = sum(a["naive slots filled"] for a in A.values())
    print(f"naive grid slots filled: {nfill:,} of {N_SLOTS*len(A):,}")

--- steps
records per participant histogram (n->participants): {39960: 3, 40320: 42, 40326: 5}
participants with records != 40320: 8  examples=['P005=40326', 'P010=40326', 'P020=39960', 'P028=39960', 'P034=40326']
duplicate minute slot (naive): records/slots=5790 participants: 8  examples=['P005=6', 'P010=1446', 'P019=1440', 'P022=1440', 'P025=1440']
naive grid gaps: records/slots=6840 participants: 7  examples=['P010=1440', 'P019=1440', 'P020=360', 'P022=1440', 'P025=1440']
out-of-order record: records/slots=4 participants: 4  examples=['P010=1', 'P019=1', 'P022=1', 'P025=1']
naive timestamp outside study grid: records/slots=0 participants: 0
naive grid slots filled: 2,009,160 of 2,016,000
--- heart_rate
records per participant histogram (n->participants): {39960: 3, 40320: 42, 40326: 5}
participants with records != 40320: 8  examples=['P005=40326', 'P010=40326', 'P020=39960', 'P028=39960', 'P034=40326']
duplicate minute slot (naive): records/slots=5790 participants: 8  examples=['P00

## 3. Timestamp formats (UTC `Z` vs naive), non-zero seconds, participant_id vs folder

In [4]:
FMT_KEYS = ["timestamp naive", "timestamp UTC 'Z' suffix", "timestamp explicit offset (not Z)", "timestamp has fractional seconds",
            "timestamp non-zero seconds", "timestamp unparseable/missing", "timestamp invalid calendar date", "participant_id != folder",
            "value null/missing", "value not an integer"]
for kind in RES:
    A = {p: RES[kind][p]["a"] for p in participants}
    print(f"--- {kind}")
    for key in FMT_KEYS:
        hit = {p: a[key] for p, a in A.items() if a[key]}
        total = sum(hit.values())
        if key == "timestamp naive":
            print(f"{key}: records={total:,} participants={len(hit)}")
        else:
            print(f"{key}: records={total:,} participants={len(hit)}", [f"{p}={n}" for p, n in sorted(hit.items(), key=lambda kv: -kv[1])[:5]] if hit else "(nothing found)")
    for key in ("Z as_label: fills an empty slot", "Z as_label: collides with naive record", "Z as_label: outside grid",
                "Z utc_converted: fills an empty slot", "Z utc_converted: collides with naive record", "Z utc_converted: outside grid"):
        print(key.replace("Z ", "Z records, interpreted ").replace(":", " ->") + ":", sum(a[key] for a in A.values()))
    zp = [p for p, a in A.items() if a["timestamp UTC 'Z' suffix"]]
    print("participants with any Z timestamp:", len(zp), sorted(zp)[:8])
    z_all = [p for p, a in A.items() if a["timestamp UTC 'Z' suffix"] == a["records"]]
    print("participants whose whole file is Z:", len(z_all))
    gaps_after = {p: a["naive grid gaps"] - a["Z utc_converted: fills an empty slot"] for p, a in A.items()}
    print("grid gaps remaining if UTC-converted Z records were accepted:", sum(gaps_after.values()), "(naive-only gaps:", sum(a['naive grid gaps'] for a in A.values()), ")")

--- steps
timestamp naive: records=2,014,950 participants=50
timestamp UTC 'Z' suffix: records=0 participants=0 (nothing found)
timestamp explicit offset (not Z): records=0 participants=0 (nothing found)
timestamp has fractional seconds: records=0 participants=0 (nothing found)
timestamp non-zero seconds: records=28 participants=5 ['P005=6', 'P010=6', 'P040=6', 'P050=6', 'P034=4']
timestamp unparseable/missing: records=0 participants=0 (nothing found)
timestamp invalid calendar date: records=0 participants=0 (nothing found)
participant_id != folder: records=0 participants=0 (nothing found)
value null/missing: records=0 participants=0 (nothing found)
value not an integer: records=0 participants=0 (nothing found)
Z records, interpreted as_label -> fills an empty slot: 0
Z records, interpreted as_label -> collides with naive record: 0
Z records, interpreted as_label -> outside grid: 0
Z records, interpreted utc_converted -> fills an empty slot: 0
Z records, interpreted utc_converted -> co

## 3b. Drill-down on anomalous files (re-reads only participants flagged above; identifiers and counts only)

Covers: duplicate minute slots (identical vs conflicting values), whole-day gaps/duplications, non-zero-second records,
and the undocumented `_unit` field seen in steps.json key sets.

In [5]:
def drill(pid, fname, vkey):
    rows = json.loads((EVENTS / pid / fname).read_bytes())
    by_slot = collections.defaultdict(list)
    sec_slots, unit_vals = [], collections.Counter()
    for r in rows:
        m = TS.match(r["timestamp"])
        if m.group(8):
            continue
        y, mo, d, hh, mi, ss = map(int, m.groups()[:6])
        slot = (date(y, mo, d).toordinal() - ORD0) * 1440 + hh * 60 + mi
        by_slot[slot].append((ss, r.get(vkey)))
        if ss:
            sec_slots.append(slot)
        if "_unit" in r:
            unit_vals[r["_unit"]] += 1
    dup = {s_: v for s_, v in by_slot.items() if len(v) > 1}
    conflicting = sum(1 for v in dup.values() if len({x[1] for x in v}) > 1)
    dup_days = collections.Counter(slot_to_ts(s_)[:10] for s_ in dup)
    gap_days = collections.Counter(slot_to_ts(s_)[:10] for s_ in range(N_SLOTS) if s_ not in by_slot)
    sec_with_neighbour = sum(1 for s_ in sec_slots if len(by_slot[s_]) > 1)
    return {"dup_slots": len(dup), "conflicting": conflicting, "dup_days": dict(dup_days), "gap_days": dict(gap_days),
            "sec_records": len(sec_slots), "sec_share_slot_with_other": sec_with_neighbour, "unit": unit_vals}

flagged = {}
for kind, fname in (("steps", "steps.json"), ("heart_rate", "heart_rate.json")):
    for pid in participants:
        a = RES[kind][pid]["a"]
        if a["duplicate minute slot (naive)"] or a["naive grid gaps"] or a["timestamp non-zero seconds"] or a["records"] != N_SLOTS or any("_unit" in ks for ks in RES[kind][pid]["keysets"]):
            flagged[(kind, pid)] = drill(pid, fname, VKEY[kind])
for (kind, pid), d in sorted(flagged.items()):
    full = lambda days: sorted(k for k, v in days.items() if v >= 1440)
    part = lambda days: {k: v for k, v in sorted(days.items()) if v < 1440}
    print(f"{kind} {pid}: dup_slots={d['dup_slots']} conflicting_values={d['conflicting']} full-day dup dates={full(d['dup_days'])} partial dup dates={part(d['dup_days'])} "
          f"| missing slots={sum(d['gap_days'].values())} full-day gap dates={full(d['gap_days'])} partial gap dates={part(d['gap_days'])} "
          f"| non-zero-second records={d['sec_records']} (sharing a minute slot with another record: {d['sec_share_slot_with_other']})")

heart_rate P005: dup_slots=5 conflicting_values=5 full-day dup dates=[] partial dup dates={'2026-01-10': 1, '2026-01-14': 2, '2026-01-23': 2} | missing slots=0 full-day gap dates=[] partial gap dates={} | non-zero-second records=6 (sharing a minute slot with another record: 6)
heart_rate P010: dup_slots=1445 conflicting_values=1297 full-day dup dates=['2026-01-10'] partial dup dates={'2026-01-12': 1, '2026-01-25': 2, '2026-01-29': 2} | missing slots=1440 full-day gap dates=['2026-01-08'] partial gap dates={} | non-zero-second records=6 (sharing a minute slot with another record: 6)
heart_rate P018: dup_slots=0 conflicting_values=0 full-day dup dates=[] partial dup dates={} | missing slots=1440 full-day gap dates=['2026-01-25'] partial gap dates={} | non-zero-second records=0 (sharing a minute slot with another record: 0)
heart_rate P019: dup_slots=1440 conflicting_values=1300 full-day dup dates=['2026-01-10'] partial dup dates={} | missing slots=1440 full-day gap dates=['2026-01-08'] p

In [6]:
unit_by_p = {}
for pid in participants:
    if any("_unit" in ks for ks in RES["steps"][pid]["keysets"]):
        unit_by_p[pid] = sum(c for ks, c in RES["steps"][pid]["keysets"].items() if "_unit" in ks)
print("steps.json records carrying an undocumented `_unit` key:", sum(unit_by_p.values()), "participants:", len(unit_by_p), sorted(unit_by_p)[:10])
print("per-participant counts of `_unit` records (histogram n->participants):", dict(sorted(collections.Counter(unit_by_p.values()).items())))
unit_domain, unit_steps = collections.Counter(), []
for pid in sorted(unit_by_p):
    rows = json.loads((EVENTS / pid / "steps.json").read_bytes())
    with_u = [r["steps"] for r in rows if "_unit" in r]
    without_u = sorted(r["steps"] for r in rows if "_unit" not in r)
    unit_domain.update(r["_unit"] for r in rows if "_unit" in r)
    unit_steps.append((pid, len(with_u), max(with_u), sorted(with_u)[len(with_u)//2], without_u[len(without_u)//2] if without_u else None, max(without_u) if without_u else None))
    del rows
print("`_unit` value domain (value -> records):", dict(unit_domain))
print("(participant, records with _unit, max steps with _unit, median steps with _unit, median steps without, max steps without):")
for row in unit_steps[:12]:
    print("  ", row)

steps.json records carrying an undocumented `_unit` key: 120963 participants: 3 ['P005', 'P033', 'P038']
per-participant counts of `_unit` records (histogram n->participants): {40320: 2, 40323: 1}
`_unit` value domain (value -> records): {'steps_per_hour': 120963}
(participant, records with _unit, max steps with _unit, median steps with _unit, median steps without, max steps without):
   ('P005', 40323, 780, 60, 180, 240)
   ('P033', 40320, 1860, 300, None, None)
   ('P038', 40320, 2460, 420, None, None)


## 4. Value ranges (steps; 40-200 bpm)

In [7]:
def quantile(counter, f):
    n = sum(counter.values()); target = f * (n - 1); acc = 0
    for v in sorted(counter):
        acc += counter[v]
        if acc > target:
            return v

steps_all = sum((RES["steps"][p]["values"] for p in participants), collections.Counter())
hr_all = sum((RES["heart_rate"][p]["values"] for p in participants), collections.Counter())
n_s, n_h = sum(steps_all.values()), sum(hr_all.values())
print(f"steps: n={n_s:,} min={min(steps_all)} p50={quantile(steps_all,.5)} p95={quantile(steps_all,.95)} p99={quantile(steps_all,.99)} p999={quantile(steps_all,.999)} max={max(steps_all)}")
print("steps < 0:", sum(c for v, c in steps_all.items() if v < 0), "| steps == 0:", steps_all[0], f"({100*steps_all[0]/n_s:.1f}%)",
      "| steps > 30 (dictionary 'expected' bound):", sum(c for v, c in steps_all.items() if v > 30),
      "| steps > 100:", sum(c for v, c in steps_all.items() if v > 100), "| steps > 200:", sum(c for v, c in steps_all.items() if v > 200))
big = {v: c for v, c in steps_all.items() if v > 1000}
print("steps > 1000: records", sum(big.values()), "| distinct values", len(big), "| records at the single most common such value", max(big.values()) if big else 0)
neg = [p for p in participants if RES["steps"][p]["a"]["min"] < 0]
show("participants with negative steps", len(neg), neg)
high = [p for p in participants if RES["steps"][p]["a"]["max"] > 200]
show("participants with steps > 200", len(high), high)
print(f"heart_rate_bpm: n={n_h:,} min={min(hr_all)} p01={quantile(hr_all,.01)} p50={quantile(hr_all,.5)} p99={quantile(hr_all,.99)} max={max(hr_all)}")
lo = sum(c for v, c in hr_all.items() if v < 40); hi = sum(c for v, c in hr_all.items() if v > 200)
print("heart_rate_bpm < 40:", lo, "| > 200:", hi, "| outside 40-200 total:", lo + hi)
oor = [p for p in participants if RES["heart_rate"][p]["a"]["min"] < 40 or RES["heart_rate"][p]["a"]["max"] > 200]
show("participants with out-of-range heart rate", len(oor), oor)

steps: n=2,014,950 min=-1 p50=5 p95=28 p99=660 p999=1320 max=1000000
steps < 0: 2 | steps == 0: 711253 (35.3%) | steps > 30 (dictionary 'expected' bound): 95902 | steps > 100: 70516 | steps > 200: 56389
steps > 1000: records 5141 | distinct values 26 | records at the single most common such value 1026
participants with negative steps: 2  examples=['P029', 'P037']
participants with steps > 200: 5  examples=['P005', 'P029', 'P033', 'P037', 'P038']
heart_rate_bpm: n=2,014,950 min=-10 p01=53 p50=69 p99=125 max=999
heart_rate_bpm < 40: 2 | > 200: 2 | outside 40-200 total: 4
participants with out-of-range heart rate: 2  examples=['P029', 'P037']


## 5. Stuck-sensor runs in heart rate

A run is consecutive records (file order) with the identical `heart_rate_bpm`. Run lengths are in records (minutes on a gap-free grid).

In [8]:
all_runs = [(p, s, n, v) for p in participants for (s, n, v) in RES["heart_rate"][p]["hr_runs"]]
print("runs of identical consecutive values with length >= 10:", len(all_runs))
for thr in (10, 15, 20, 30, 60, 120, 240, 480):
    sel = [r for r in all_runs if r[2] >= thr]
    print(f"  runs >= {thr:>3} min: runs={len(sel):>4} participants={len({r[0] for r in sel}):>3} minutes_in_runs={sum(r[2] for r in sel):>7,}")
print("run length histogram (length bucket -> runs):", dict(sorted(collections.Counter((r[2] // 30) * 30 for r in all_runs).items())))
stuck = [r for r in all_runs if r[2] >= 60]
longest = sorted(stuck, key=lambda r: -r[2])[:10]
print("longest runs (participant, start-minute, length-min):", [(p, slot_to_ts(s) if s is not None else None, n) for p, s, n, v in longest])
print("participants with a run >= 60 min:", len({r[0] for r in stuck}), "| runs per participant:", dict(sorted(collections.Counter(r[0] for r in stuck).items())))
print("distinct stuck values among runs >= 60 min:", len({r[3] for r in stuck}), "| runs >=60 whose value is within 40-200:", sum(1 for r in stuck if 40 <= r[3] <= 200))
tot_hr = sum(RES["heart_rate"][p]["a"]["records"] for p in participants)
print(f"share of all heart-rate records inside runs >= 60 min: {100*sum(r[2] for r in stuck)/tot_hr:.3f}%")
# Heart-rate timestamps identical to steps timestamps?
same = sum(1 for p in participants if RES["steps"][p]["ts_hash"] == RES["heart_rate"][p]["ts_hash"])
print("participants whose steps and heart_rate timestamp sequences are identical:", same, "of", len(participants))

runs of identical consecutive values with length >= 10: 2
  runs >=  10 min: runs=   2 participants=  2 minutes_in_runs=    360
  runs >=  15 min: runs=   2 participants=  2 minutes_in_runs=    360
  runs >=  20 min: runs=   2 participants=  2 minutes_in_runs=    360
  runs >=  30 min: runs=   2 participants=  2 minutes_in_runs=    360
  runs >=  60 min: runs=   2 participants=  2 minutes_in_runs=    360
  runs >= 120 min: runs=   2 participants=  2 minutes_in_runs=    360
  runs >= 240 min: runs=   0 participants=  0 minutes_in_runs=      0
  runs >= 480 min: runs=   0 participants=  0 minutes_in_runs=      0
run length histogram (length bucket -> runs): {180: 2}
longest runs (participant, start-minute, length-min): [('P003', '2026-01-09T04:44', 180), ('P042', '2026-01-24T09:16', 180)]
participants with a run >= 60 min: 2 | runs per participant: {'P003': 1, 'P042': 1}
distinct stuck values among runs >= 60 min: 2 | runs >=60 whose value is within 40-200: 2
share of all heart-rate reco

## Runtime

In [9]:
tr = sum(RES[k][p]["t_read"] for k in RES for p in participants); tp = sum(RES[k][p]["t_parse"] for k in RES for p in participants)
print(f"per-minute JSON read {tr:.2f}s + parse {tp:.2f}s over 100 files; total notebook runtime: {time.perf_counter() - T0:.1f} s (harness limit 600 s)")

per-minute JSON read 0.07s + parse 1.31s over 100 files; total notebook runtime: 96.5 s (harness limit 600 s)
